<a href="https://colab.research.google.com/github/Leafarex/computo-inteligente-Rafael-Negrete-Leyva/blob/Tarea-de-2%2F5%2F2026/pipeline_overwatch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Overwatch 2 Team Composition Analysis
## Exploratory Data Analysis and Machine Learning Pipeline


## Introduction

In this notebook, we perform an Exploratory Data Analysis (EDA) on an Overwatch 2 hero statistics dataset obtained from Kaggle.  
Based on the findings from the EDA, we construct a preprocessing and machine learning pipeline to estimate the probability of winning a match given a team composition of five heroes.
link of the data https://www.kaggle.com/datasets/mykhailokachan/overwatch-2-statistics/data

## Libraries

The following libraries are used throughout the notebook:
- **pandas** and **numpy** for data manipulation
- **matplotlib** for data visualization
- **scikit-learn** for preprocessing, pipeline construction, and machine learning modeling


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression


## Dataset Description

The dataset contains hero-level performance metrics such as win rate, damage, healing, eliminations, and deaths, grouped by hero and skill tier.  
These statistics represent historical averages and are used as the basis for constructing team-level features.


This code loads the Overwatch 2 dataset from a CSV file into a pandas DataFrame.


In [ ]:
DATA_PATH = "ow2_quickplay_heroes_stats__2023-05-06.csv"

df = pd.read_csv(DATA_PATH)
df.head()


This section displays general information about the dataset, including column names, data types, and missing values.


In [ ]:
df.info()


A subset of relevant columns is selected to focus the analysis on general hero performance statistics.


In [ ]:
selected_columns = [
    "Hero",
    "Skill Tier",
    "Win Rate, %",
    "Eliminations / 10min",
    "Damage / 10min",
    "Healing / 10min",
    "Deaths / 10min",
    "Final Blows / 10min"
]

df_selected = df[selected_columns].copy()
df_selected.head()


Descriptive statistics are computed to summarize the numerical features of the dataset.



In [ ]:
df_selected.describe().T


Histograms are used to visualize the distributions of the numerical features and identify differences in scale.


In [ ]:
numerical_cols = [
    "Win Rate, %",
    "Eliminations / 10min",
    "Damage / 10min",
    "Healing / 10min",
    "Deaths / 10min",
    "Final Blows / 10min"
]

df_selected[numerical_cols].hist(figsize=(12, 8), bins=20)
plt.tight_layout()
plt.show()


This code checks for missing values in the selected columns.


In [ ]:
df_selected.isna().sum()


A dictionary is defined to map each hero to its corresponding role: tank, damage, or support.


In [ ]:
HERO_ROLE = {
    "Ana": "sup", "Ashe": "dps", "Baptiste": "sup", "Bastion": "dps",
    "Brigitte": "sup", "Cassidy": "dps", "D.Va": "tank", "Doomfist": "tank",
    "Echo": "dps", "Genji": "dps", "Hanzo": "dps", "Junker Queen": "tank",
    "Junkrat": "dps", "Kiriko": "sup", "Lucio": "sup", "Mei": "dps",
    "Mercy": "sup", "Moira": "sup", "Orisa": "tank", "Pharah": "dps",
    "Ramattra": "tank", "Reaper": "dps", "Reinhardt": "tank",
    "Roadhog": "tank", "Sigma": "tank", "Sojourn": "dps",
    "Soldier: 76": "dps", "Sombra": "dps", "Symmetra": "dps",
    "Torbjorn": "dps",
}

df_lookup = df_selected.copy()
df_lookup["Role"] = df_lookup["Hero"].map(HERO_ROLE)


Numerical columns are explicitly converted to numeric types to avoid issues during aggregation.


In [ ]:
stats_cols = [
    "Win Rate, %",
    "Eliminations / 10min",
    "Damage / 10min",
    "Healing / 10min",
    "Deaths / 10min",
    "Final Blows / 10min"
]

for c in stats_cols:
    df_lookup[c] = pd.to_numeric(df_lookup[c], errors="coerce")

tiers = sorted(df_lookup["Skill Tier"].dropna().unique().tolist())


Functions are defined to build a team-level dataset by aggregating hero statistics.


In [ ]:
def get_hero_stats(hero_name, skill_tier):
    row = df_lookup[(df_lookup["Hero"] == hero_name) & (df_lookup["Skill Tier"] == skill_tier)]
    return row.iloc[0]

def build_team_row(skill_tier, heroes_5):
    team_stats = [get_hero_stats(h, skill_tier) for h in heroes_5]
    tmp = pd.DataFrame(team_stats)

    return pd.DataFrame([{
        "avg_winrate": tmp["Win Rate, %"].mean(),
        "avg_elims": tmp["Eliminations / 10min"].mean(),
        "avg_damage": tmp["Damage / 10min"].mean(),
        "avg_healing": tmp["Healing / 10min"].mean(),
        "avg_deaths": tmp["Deaths / 10min"].mean(),
        "avg_final_blows": tmp["Final Blows / 10min"].mean(),
        "hero_1": heroes_5[0],
        "hero_2": heroes_5[1],
        "hero_3": heroes_5[2],
        "hero_4": heroes_5[3],
        "hero_5": heroes_5[4],
    }])


The dataset is split into features (X) and target (y) for model training.


In [ ]:
# Generate a sample team_df for demonstration purposes
# In a real scenario, this DataFrame would likely come from a dataset of actual matches

team_compositions = [
    ("All", ["D.Va", "Ashe", "Junkrat", "Ana", "Lucio"], 1), # Example team 1 (win)
    ("All", ["Sigma", "Reaper", "Soldier: 76", "Mercy", "Moira"], 0), # Example team 2 (loss)
    ("All", ["Orisa", "Echo", "Genji", "Baptiste", "Kiriko"], 1), # Example team 3 (win)
    ("All", ["Reinhardt", "Hanzo", "Cassidy", "Ana", "Mercy"], 0), # Example team 4 (loss)
    ("All", ["D.Va", "Mei", "Sojourn", "Kiriko", "Brigitte"], 1), # Example team 5 (win)
    ("All", ["Junker Queen", "Pharah", "Sombra", "Baptiste", "Lucio"], 0), # Example team 6 (loss)
]

team_rows = []
for tier, heroes, team_win_outcome in team_compositions:
    row = build_team_row(tier, heroes)
    row["team_win"] = team_win_outcome
    team_rows.append(row)

team_df = pd.concat(team_rows, ignore_index=True)

print("Sample team_df created successfully:")
print(team_df.head())

In [ ]:
X = team_df.drop(columns=["team_win"])
y = team_df["team_win"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

To further explore the relationships between numerical variables, a correlation matrix is computed.
This analysis helps identify linear relationships between features and the target-related statistics.


In [ ]:
corr_cols = [
    "Win Rate, %",
    "Eliminations / 10min",
    "Damage / 10min",
    "Healing / 10min",
    "Deaths / 10min",
    "Final Blows / 10min"
]

corr_matrix = df_selected[corr_cols].corr()

corr_matrix


Based on the exploratory data analysis, several observations can be made:

- Win rate shows a positive correlation with damage and eliminations per 10 minutes, indicating that offensive output is associated with higher chances of winning.
- Healing per 10 minutes exhibits moderate correlation with win rate, suggesting that team sustainability also plays an important role.
- Deaths per 10 minutes tend to have a negative correlation with win rate, which aligns with expected gameplay behavior.
- Some features present different value scales, justifying the use of feature scaling during preprocessing.

These insights directly motivate the feature selection and preprocessing strategies used in the machine learning pipeline.


In [ ]:
corr_cols = [
    "Win Rate, %",
    "Eliminations / 10min",
    "Damage / 10min",
    "Healing / 10min",
    "Deaths / 10min",
    "Final Blows / 10min"
]

corr_matrix = df_selected[corr_cols].corr()

corr_matrix


A preprocessing pipeline is created to handle numerical scaling and categorical encoding.


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, [
        "avg_winrate","avg_elims","avg_damage","avg_healing","avg_deaths","avg_final_blows"
    ]),
    ("cat", categorical_pipeline, [
        "hero_1","hero_2","hero_3","hero_4","hero_5"
    ])
])


The preprocessing pipeline is combined with a logistic regression classifier and trained using the training dataset.


In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

The trained model is used to generate predictions and estimate win probabilities for a selected team.


In [ ]:
y_pred = model.predict(X_test)
y_pred[:10]


A heatmap is used to visualize the correlation matrix, making it easier to identify strong positive or negative relationships between variables.


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues", interpolation="nearest")
plt.colorbar()

classes = np.unique(y_test)

plt.xticks(range(len(classes)), classes)
plt.yticks(range(len(classes)), classes)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j],
                 ha="center", va="center",
                 color="white" if cm[i, j] > cm.max() / 2 else "black")

plt.tight_layout()
plt.show()


In this section, an interactive interface is created to allow the user to select a skill tier and a team composition of five heroes.
The selected heroes will later be used to compute team-level statistics and predict the probability of winning a match.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output


To better understand the model behavior, the predicted labels are compared against the actual labels from the test dataset.
This comparison allows a direct inspection of correct and incorrect predictions.


In [ ]:
results_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

results_df.head(10)


This section summarizes how many predictions were correct and incorrect by comparing the actual and predicted values.


In [ ]:
results_df["Correct"] = results_df["Actual"] == results_df["Predicted"]
results_df["Correct"].value_counts()


A simple bar chart is used to visualize the distribution of correct and incorrect predictions.


In [ ]:
results_df["Correct"] = results_df["Actual"] == results_df["Predicted"]
results_df["Correct"].value_counts()


This code retrieves the list of available heroes and skill tiers from the dataset, which will be used to populate the selection widgets.


In [ ]:
heroes = sorted(df_lookup["Hero"].dropna().unique().tolist())
tiers


A helper function is defined to filter heroes based on the selected skill tier and their predefined role.
This ensures that only valid heroes are displayed for each team position.


In [ ]:
def available_heroes(skill_tier, role):
    sub = df_lookup[(df_lookup["Skill Tier"] == skill_tier) & (df_lookup["Role"] == role)]
    return sorted(sub["Hero"].dropna().unique().tolist())


A dropdown widget is created to allow the user to select the skill tier for the team composition.


In [ ]:
tier_dd = widgets.Dropdown(
    options=tiers,
    description='Skill Tier:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px')
)

def hero_dropdown(label, role):
    dd = widgets.Dropdown(
        options=[],
        description=label,
        style={"description_width": "initial"},
        layout=widgets.Layout(width="450px")
    )

    def update(*args):
        opts = available_heroes(tier_dd.value, role)
        dd.options = opts
        if opts:
            dd.value = opts[0]

    tier_dd.observe(update, names="value")
    update() # Initial update to populate hero dropdowns
    return dd

tank_dd = hero_dropdown("Tank:", "tank")
dps1_dd = hero_dropdown("DPS 1:", "dps")
dps2_dd = hero_dropdown("DPS 2:", "dps")
sup1_dd  = hero_dropdown("Support 1:", "sup")
sup2_dd  = hero_dropdown("Support 2:", "sup")

display(tier_dd, tank_dd, dps1_dd, dps2_dd, sup1_dd, sup2_dd)

A button is added to trigger the computation of team statistics and the prediction of the match outcome.
The results are displayed dynamically when the button is pressed.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

output = widgets.Output()
btn = widgets.Button(description="Calculate Team Statistics", button_style="success")

display(btn, output)

This function aggregates the selected heroes into a single team, computes average team statistics,
and uses the trained machine learning model to estimate the probability of winning a match.


In [ ]:
def on_click(_):
    with output:
        clear_output()

        heroes_5 = [
            tank_dd.value,
            dps1_dd.value,
            dps2_dd.value,
            sup1_dd.value,
            sup2_dd.value
        ]

        X_one = build_team_row(tier_dd.value, heroes_5)

        summary = X_one[[
            "avg_winrate",
            "avg_elims",
            "avg_damage",
            "avg_healing",
            "avg_deaths",
            "avg_final_blows"
        ]]

        win_pred = int(model.predict(X_one)[0])
        win_prob = float(model.predict_proba(X_one)[0, 1])

        print("=== Selected Team ===")
        print("Skill Tier:", tier_dd.value)
        print("Heroes:", heroes_5)

        print("\n=== Team Average Statistics ===")
        display(summary)

        print("\n=== Match Outcome Prediction ===")
        print("Predicted Win (0/1):", win_pred)
        print("Win Probability:", round(win_prob, 4))


The button click event is linked to the prediction function, enabling the interactive workflow.


In [ ]:
btn.on_click(on_click)
